<a href="https://colab.research.google.com/github/alijon070/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch bitsandbytes datasets peft trl

In [2]:


from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
from datasets import load_dataset
from peft import LoraModel, LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import torch

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)
# print('Special tokens ids:', tokenizer.all_special_ids)

# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

In [ ]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is?"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# print(model)
first_block= model.model.layers[0]
print(first_block)
print('=========')
print(first_block.self_attn)
print('=========')
print(model.config)

In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

## datasets library | load dataset
### * instruction tuning

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

dataset

In [ ]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_text}"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}"
    )

# generate_prompt(dataset[0])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)



In [ ]:
dataset[0]['text']

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42) # seed keeps the order of array

In [ ]:
dataset

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory

In [ ]:
config = LoraConfig(
    r=8, # rank
    lora_alpha=16, # scaling
    lora_dropout=0.05,
    task_type="CAUSAL_LM", # predict next token
    target_modules=["q_proj", "v_proj"],
)

In [ ]:
model = get_peft_model(model, config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
# QLora
# Lora

In [ ]:
training_args = SFTConfig(
    output_dir="./jaydari_gpt",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, #VRAM
    num_train_epochs=2, # overfit
    logging_steps=20,
    save_strategy="epoch",
    bf16=False, # GPU
    fp16=False, # memory
    report_to="none",
    dataset_text_field="text"
)

In [ ]:

# SFTTrainer vs Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x['text'],
    args=training_args
)

trainer.train()

model.save_pretrained('jaydari_gpt')
tokenizer.save_pretrained('jaydari_gpt')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_base = AutoTokenizer.from_pretrained(base_model_id)
model_base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    )

model_base.eval()

In [ ]:
prompt = "Explain what machine learning is in simple words."

inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

with torch.no_grad():
   output_base = model_base.generate(
    **inputs_base,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.7
)

print("=== Base Model ===")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))



In [ ]:
FINE-TUNED MODEL INFERENCE

In [ ]:
!pip install -U torchao

from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "jaydari_gpt"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

In [ ]:
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

inputs_ft = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
   output_ft = model.generate(
    **inputs_ft,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.7
)

print("=== FINE-TUNED Model ===")
print(tokenizer.decode(output_ft[0], skip_special_tokens=True))

In [ ]:
!zip -r jaydari_gpt.zip jaydari_gpt

In [ ]:
from google.colab import files
files.download('jaydari_gpt.zip')

In [ ]:
###########################
#    MODEL TESTING        #
###########################



In [3]:
!pip install -q langdetect

In [ ]:
!unzip jaydari_gpt.zip

In [6]:
!pip install -U torchao

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/jaydari_gpt"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    dtype=torch.float16
)

# Load fine-tuned adapter
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 17.5 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [7]:
from langdetect import detect_langs

def ask_jaydari(prompt: str, max_new_tokens=256):
  try:
    langs = detect_langs(prompt)
    lang_code = langs[0].lang
    confidence =langs[0].prob
    print(f"Detected lang: {lang_code} ({confidence:.2f})")
  except:
    lang_code = "unknown"
    confidence = 0.0

  # Block only if we're confident it's NOT English.
  if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:
    return "Sorry, you should ask a question only in English."

  # Proceed with generation
  full_prompt = f"### Instruction:\n{prompt}\n\n### Response:\n"
  inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

  outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=True,
      temperature=0.7,
      eos_token_id=tokenizer.eos_token_id,
  )

  decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return decoded[len(full_prompt):].strip()

In [12]:
ask_jaydari('qoyxona nima edi qaysi til bu?')

Detected lang: so (1.00)


'Sorry, you should ask a question only in English.'